# Glass Defect Inspector: local audit companion
Read-only companion to `scripts/audit_inspector.py`. No source dataset or benchmark is modified. Counts below are from fully decoded local images and YOLO labels; pHash candidates require visual/source-lineage review.

In [ ]:
from pathlib import Path
import json
from collections import Counter
root = Path.cwd() if Path('reports/inspection/inspector_audit.json').exists() else Path.cwd().parent
audit = json.loads((root/'reports/inspection/inspector_audit.json').read_text())
rows = [json.loads(line) for line in (root/'reports/inspection/inspector_data_manifest.jsonl').read_text().splitlines()]
Counter(row['split'] for row in rows)

In [ ]:
{split: {'images': sum(r['split']==split for r in rows), 'boxes': sum(len(r['boxes']) for r in rows if r['split']==split), 'resolutions': dict(Counter(tuple(r['size']) for r in rows if r['split']==split)), 'label_or_decode_issues': sum(bool(r['issues']) for r in rows if r['split']==split)} for split in ['train','val','test']}

In [ ]:
{'exact_cross_split_groups': len(audit['cross_split_exact']), 'pHash_candidates': audit['cross_split_near_count'], 'pHash_by_split': audit['cross_split_near_by_split'], 'capture_groups': audit['capture_lineage']['cross_split_capture_groups'], 'coco_variants': [{k:v for k,v in c.items() if k in ['path','images','boxes','resolutions','inferred_raw_offsets']} for c in audit['coco']]}

Interpretation: no duplicate encoded bytes between splits; whole-frame pHash has low specificity on a fixed camera scene. Capture-family overlap suggests augmentation leakage and needs source confirmation. Historical COCO annotations belong to a narrower ROI than the current processing configuration, so box counts and benchmark lineage cannot be treated as interchangeable. Missing files/dependencies are reported separately; zero detections require technician REVIEW.